In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/pytorch)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

In [ ]:
from typing import Any

import torch as tr
import torch.optim as optim
import torch.autograd as autograd

import import_ipynb
from approx import approx # type: ignore

In [ ]:
def count(vector: tr.Tensor) -> tuple[tr.Tensor, tr.Tensor]:
    """Return a histogram of a vector of integers as a tuple of (unique_values, counts)."""

    if isinstance(vector, tr.Tensor) == False:
        vector = tr.tensor(vector, dtype=tr.long)

    return vector.unique(sorted=True, return_counts=True)


def test_count():
    actual = count([1, 1, 2, 1, 2, 3, 1, 2, 3, 4, 1, 2, 3, 4, 5])
    expected = (tr.tensor([1, 2, 3, 4, 5]), tr.tensor([5, 4, 3, 2, 1]))

    assert actual[0] == approx(expected[0])
    assert actual[1] == approx(expected[1])

if __name__ == "__main__":
    test_count()

In [ ]:
def repeat(fn, times=10, reduction=tr.mean):
    return reduction(tr.tensor([fn() for _ in range(times)]))

In [ ]:
def test_model_sgd_2(model, x: tr.Tensor, y: tr.Tensor, epochs=500, lr=0.1):
    optimizer = optim.SGD(model.parameters(), lr=lr)

    for _ in range(epochs):
        optimizer.zero_grad()

        if hasattr(model, 'CUSTOM_GRAD'):
            loss = model(x, y)
        else:
            p = model(x)
            loss = model.loss(p, y)

        loss.backward()
        optimizer.step()

    with tr.no_grad():
        return tr.isclose(model.predict(x), y, atol=0.01, rtol=0.0).float().mean().item()

In [ ]:
def test_checkgrad_2(FN, 
                     S: int, 
                     FI: int, 
                     FO: int, 
                     prec=0.001, 
                     fX = lambda x:x, 
                     fY = lambda y:y):
    W = tr.randn(FI, FO, dtype=tr.float64, requires_grad=True)
    b = tr.randn(FO, dtype=tr.float64, requires_grad=True)
    
    x = fX(tr.randn(S, FI, dtype=tr.float64, requires_grad=False))
    y = fY(tr.randn(S, FO, dtype=tr.float64, requires_grad=False))
    
    assert autograd.gradcheck(FN.apply, (x, W, b, y), eps=prec, atol=prec, rtol=0)


In [ ]:
def test_compare_2(expected_model_type: type, 
                          actual_model_type: type, 
                          S: int, 
                          FI: int, 
                          FO: int, 
                          E=5, 
                          fX = lambda x:x, 
                          fY = lambda y:y) -> None:
    expected_model = expected_model_type(FI, FO)
    actual_model = actual_model_type(FI, FO, expected_model.linear.weight, expected_model.linear.bias)

    w = tr.randn(FI, FO, dtype=tr.float32)
    b = tr.randn(S, FO, dtype=tr.float32)
    
    x = fX(tr.randn(S, FI, dtype=tr.float32))
    y = fY((x @ w + b >= 0.5).float())

    expected_optimizer = optim.SGD(expected_model.parameters(), lr=0.1)
    actual_optimizer = optim.SGD(actual_model.parameters(), lr=0.1)

    for _ in range(E):
        expected_optimizer.zero_grad()
        actual_optimizer.zero_grad()

        if hasattr(expected_model, 'CUSTOM_GRAD'):
            expected_loss = expected_model(x, y)
        else:
            expected_loss = expected_model.loss(expected_model(x), y)

        if hasattr(actual_model, 'CUSTOM_GRAD'):
            actual_loss = actual_model(x, y)
        else:
            actual_loss = actual_model.loss(actual_model(x), y)

        expected_loss.backward()
        actual_loss.backward()

        expected_optimizer.step()
        actual_optimizer.step()

        with tr.no_grad():
            assert expected_loss == approx(actual_loss)

    return (expected_model, actual_model)